# Metis v0.1 — Colab T4 / CUDA benchmark

This notebook trains the PyTorch Transformer on the deterministic synthetic dataset, then measures the same trained checkpoint on CUDA and CPU. It saves held-out quality metrics, warm batch-1 TTD, throughput at batches 1/8/32/128, hardware details, and the model checkpoint.

In Colab, select **Runtime → Change runtime type → T4 GPU**. If the source is not already in `/content/metis`, upload exactly one zip of this repository with `pyproject.toml` at its root. From the repository, make one with `zip -r metis-source.zip pyproject.toml README.md metis training benchmarks docs -x '*/__pycache__/*'`. The notebook records the accelerator name and labels non-T4 CUDA runs clearly. Run cells from top to bottom; the last cell downloads a zip of the results.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

PROJECT = Path('/content/metis')
if not (PROJECT / 'pyproject.toml').is_file():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError('Upload exactly one zip containing the Metis source tree.')
    staging = Path('/content/metis-upload')
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        for member in archive.infolist():
            destination = (staging / member.filename).resolve()
            if not destination.is_relative_to(staging.resolve()):
                raise RuntimeError(f'Unsafe path in source archive: {member.filename}')
        archive.extractall(staging)
    candidates = list(staging.rglob('pyproject.toml'))
    if len(candidates) != 1:
        raise RuntimeError('The archive must contain one Metis pyproject.toml.')
    source_root = candidates[0].parent
    shutil.copytree(source_root, PROJECT, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.git', '.venv', 'runs', '__pycache__'))
os.chdir(PROJECT)
# Keep the CUDA-enabled PyTorch wheel supplied by the Colab runtime.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
print('Metis source:', PROJECT)


In [ ]:
import json
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Set the Colab runtime to a GPU and rerun.')
gpu_name = torch.cuda.get_device_name(0)
hardware = {
    'platform': platform.platform(),
    'python': sys.version,
    'pytorch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu_name': gpu_name,
    'is_t4': 'T4' in gpu_name.upper(),
}
print(json.dumps(hardware, indent=2))
if not hardware['is_t4']:
    print('NOTE: This is a CUDA run, but the detected accelerator is not a T4.')

In [ ]:
from pathlib import Path
from metis.model import ModelConfig
from training.train import train

RUN_DIR = PROJECT / 'runs' / 'colab-t4-transformer'
record = train(
    config=ModelConfig(family='transformer', embedding_dim=64, layers=2,
                       attention_heads=4, ffn_multiplier=2, count_encoding='scalar'),
    output_dir=RUN_DIR,
    size=12000, dataset_seed=17, training_seed=23, epochs=12,
    batch_size=128, learning_rate=0.001, device='cuda',
)
record['colab_hardware'] = hardware
(RUN_DIR / 'colab_hardware.json').write_text(json.dumps(hardware, indent=2) + '\n')
print(json.dumps({
    'parameters': record['parameter_count'],
    'test_decision_metrics': record['test_metrics']['decision'],
    'heuristic_comparison': record['test_comparisons']['heuristic'],
    'choice_calibration': record['choice_calibration'],
    'training_cuda_ttd': record['ttd']['ttd'],
}, indent=2))

In [ ]:
from benchmarks.ttd import benchmark as benchmark_ttd
from benchmarks.throughput import benchmark as benchmark_throughput
from metis.model import ModelConfig, build_model

checkpoint = torch.load(RUN_DIR / 'model.pt', map_location='cpu', weights_only=False)
model = build_model(ModelConfig(**checkpoint['model_config']))
model.load_state_dict(checkpoint['state_dict'])
model.choice_temperature = checkpoint.get('choice_temperature', 1.0)

runtime_results = {}
for device in ('cuda', 'cpu'):
    ttd = benchmark_ttd(model, device=device, warmup=100, iterations=1000)
    ttd.update({'device': device, 'backend': 'pytorch', 'gpu_name': gpu_name,
                'pytorch_version': torch.__version__, 'platform': platform.platform(),
                'model': checkpoint['model_config'],
                'parameter_count': sum(p.numel() for p in model.parameters())})
    throughput = benchmark_throughput(model, device=device, iterations=100)
    runtime_results[device] = {'ttd': ttd, 'throughput': throughput}
    (RUN_DIR / f'ttd_{device}.json').write_text(json.dumps(ttd, indent=2) + '\n')
    throughput_record = {'device': device, 'backend': 'pytorch', 'gpu_name': gpu_name,
                        'model': checkpoint['model_config'], 'results': throughput}
    (RUN_DIR / f'throughput_{device}.json').write_text(
        json.dumps(throughput_record, indent=2) + '\n')

comparison = {'hardware': hardware, 'runtime_results': runtime_results}
(RUN_DIR / 'cuda_cpu_comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
print(json.dumps({device: {
    'ttd_p50_ms': result['ttd']['ttd']['p50_ms'],
    'ttd_p95_ms': result['ttd']['ttd']['p95_ms'],
    'ttd_p99_ms': result['ttd']['ttd']['p99_ms'],
    'decisions_per_second': {batch: round(values['decisions_per_second'])
                             for batch, values in result['throughput'].items()},
} for device, result in runtime_results.items()}, indent=2))

In [ ]:
from google.colab import files

bundle = Path('/content/metis-colab-t4-artifacts.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file() and path.suffix in {'.json', '.pt'}:
            archive.write(path, arcname=f'{RUN_DIR.name}/{path.name}')
print('Saved artifacts:', bundle)
files.download(str(bundle))